# Notebook 11 vectorized execution guards

Purpose: reduce repeated per-image GPU synchronization while preserving the learned operator, predictor, EMA teacher, loss and gradients. Position shape/dtype/range/nonempty/uniqueness contracts are checked with batched tensor operations. Context/target overlap is checked by batched boolean equality. Numerical forward code is copied faithfully from the frozen original JEPA forward.

Inputs and outputs match the original APIs. Tagged imports do not install hooks; the Notebook 11 runner calls `install_execution_optimizations(root)` before constructing a model. The installer also refreshes the position-validation alias in any already loaded Notebook 11 encoder module. It does not alter positional feature generation, precision, finite-gradient handling or old notebook files. Expected runtime: less than one minute. No study starts here. Checkpoints and exact resume belong to the consuming workflow under `checkpoints/<run_id>/`. Validation: `tests/11_Execution_Optimization_Validation.ipynb`.


In [ ]:
from pathlib import Path
import json, os
ROOT=Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_bootstrap={}
for _cell in json.loads((ROOT/"fmm_jepa/bootstrap.ipynb").read_text())["cells"]:
    if _cell["cell_type"]=="code" and "ahr-export" in _cell.get("metadata",{}).get("tags",[]):
        exec("".join(_cell["source"]), _bootstrap)
_bootstrap["install_notebook_package"](ROOT)


In [ ]:
"""Execution-only vectorized position/mask guards; numerical model is unchanged."""
from pathlib import Path
import sys
import torch
from torch.nn import functional as F
import fmm_jepa.models as MODELS

BACKEND = "vectorized-jepa-position-guards-v1"


def vectorized_validate_positions(indices,valid,batch_size,count):
    if indices.ndim!=2 or indices.shape[0]!=batch_size or indices.shape!=valid.shape:
        raise ValueError('indices/valid must have matching [B,V] shapes')
    if indices.dtype!=torch.long or valid.dtype!=torch.bool:
        raise TypeError('indices must be long and valid must be bool')
    if not bool(valid.any(dim=1).all()):
        raise ValueError('each sample requires at least one valid position')
    if bool((valid & ((indices<0)|(indices>=count))).any()):
        raise ValueError('valid position outside original patch grid')
    ordered=torch.where(valid,indices,torch.full_like(indices,count)).sort(dim=1).values
    if bool(((ordered[:,1:]==ordered[:,:-1]) & (ordered[:,1:]<count)).any()):
        raise ValueError('valid positions must be unique within each sample/block')


def vectorized_jepa_forward(self,images,context_indices,context_valid,target_indices,target_valid):
    # Faithful copy of original JEPA.forward. Only the validation/overlap guard differs.
    target_indices=[target_indices] if isinstance(target_indices,torch.Tensor) else list(target_indices)
    target_valid=[target_valid] if isinstance(target_valid,torch.Tensor) else list(target_valid)
    if not target_indices or len(target_indices)!=len(target_valid):
        raise ValueError('nonempty matching target index and valid lists required')
    grid=(images.shape[-2]//self.context.patch_size,images.shape[-1]//self.context.patch_size)
    for indices,valid in zip(target_indices,target_valid):
        vectorized_validate_positions(indices,valid,images.shape[0],grid[0]*grid[1])
        # Equality preserves original per-sample isin semantics, including padding.
        overlap=((context_indices[:,:,None]==indices[:,None,:])
                 & context_valid[:,:,None] & valid[:,None,:])
        if bool(overlap.any()):
            raise ValueError('context positions must exclude the union of all target blocks')
    context_features=self.context(images,context_indices,context_valid)
    with torch.no_grad():
        complete_target=self.target(images)
        normalized_target=F.layer_norm(complete_target,(complete_target.shape[-1],))
        targets=[normalized_target.gather(1,torch.where(v,i,0).unsqueeze(-1).expand(-1,-1,self.context.dim))*v.unsqueeze(-1)
                 for i,v in zip(target_indices,target_valid)]
    predictions=[self.predictor(context_features,context_indices,context_valid,indices,valid,grid)
                 for indices,valid in zip(target_indices,target_valid)]
    loss=MODELS.normalized_jepa_loss(predictions,targets,target_valid)
    metrics={'loss':float(loss.detach())}
    metrics.update(MODELS._representation_metrics(context_features,context_valid,'context'))
    metrics.update(MODELS._representation_metrics(torch.cat(targets,dim=1),torch.cat(target_valid,dim=1),'target'))
    metrics.update(MODELS._representation_metrics(torch.cat(predictions,dim=1),torch.cat(target_valid,dim=1),'prediction'))
    branch_stats=[getattr(block.mixer,'last_metrics',{}) for block in self.context.blocks]
    for name in ('near_norm','far_norm'):
        values=[float(stats[name]) for stats in branch_stats if name in stats]
        if values:metrics[name]=sum(values)/len(values)
    return {'loss':loss,'prediction':predictions,'target':targets,'context_features':context_features,'metrics':metrics}


def _refresh_encoder_aliases(root,validate):
    expected=str((Path(root)/'experiments/11_manifold_long/encoder.ipynb').resolve())
    names=[]
    for name,module in list(sys.modules.items()):
        filename=getattr(module,'__file__',None)
        if filename and str(Path(filename).resolve())==expected and hasattr(module,'validate_positions'):
            module.validate_positions=validate;names.append(name)
    return sorted(names)


def install_execution_optimizations(root):
    """Install source-bound guards before model construction; no weight/RNG updates."""
    record=getattr(MODELS,'_fmm11_execution_optimizations',None)
    if record is None:
        record={'original_validate_positions':MODELS.validate_positions,'original_forward':MODELS.JEPA.forward,
                'validate_positions':vectorized_validate_positions,'forward':vectorized_jepa_forward}
        MODELS._fmm11_execution_optimizations=record
    MODELS.validate_positions=record['validate_positions'];MODELS.JEPA.forward=record['forward']
    aliases=_refresh_encoder_aliases(root,record['validate_positions'])
    return {'status':'INSTALLED','backend':BACKEND,'guards_only':True,'model_math_unchanged':True,
            'position_features_unchanged':True,'no_precision_changes':True,'patched_encoder_aliases':aliases,
            'changed_guards':['batched position uniqueness','batched context/target overlap']}


In [ ]:
print({"backend":BACKEND,"installed":False,"training_started":False})